In [1]:
from pathlib import Path

import sqlite3
%pip install pandas
import pandas as pd

db_path = Path("cohort.db").resolve()

print(db_path)
print("Database exists:", db_path.exists())

con = sqlite3.connect(db_path)

Note: you may need to restart the kernel to use updated packages.


/Users/sofia/Desktop/hds-practical/Lab4/cohort.db
Database exists: True


In [2]:
def q(sql):
    return pd.read_sql(sql, con)

q("SELECT name FROM sqlite_master WHERE type='table'")

,name
0,patients
1,samples
2,genes
3,mutations


### Patient and sample features

The patient table stays on the left. The `LEFT JOIN` keeps all 566 patients. Missing sex, age, or stage stays as `NA`. `NA` means the value was not recorded.

In [3]:
patient_features = q("""
SELECT
    p.patient_id,
    p.sex,
    p.age_at_diagnosis,
    p.stage,
    s.tissue_source_site,
    s.tmb_nonsynonymous
FROM patients AS p
LEFT JOIN samples AS s
  ON s.patient_id = p.patient_id
ORDER BY p.patient_id
""")

print("Patient-feature rows:", len(patient_features))
print(
    "Unique patients:",
    patient_features["patient_id"].nunique()
)

patient_features.head(10)

Patient-feature rows: 566
Unique patients: 566


,patient_id,sex,age_at_diagnosis,stage,tissue_source_site,tmb_nonsynonymous
0,TCGA-05-4244,male,70.0,IV,Indivumed,6.400000
1,TCGA-05-4249,male,67.0,IB,Indivumed,10.000000
2,TCGA-05-4250,female,79.0,IIIA,Indivumed,10.500000
3,TCGA-05-4382,male,68.0,IB,Indivumed,51.733333
4,TCGA-05-4384,male,66.0,IIIA,Indivumed,3.966667
5,TCGA-05-4389,male,70.0,IA,Indivumed,7.033333
6,TCGA-05-4390,female,58.0,IB,Indivumed,17.333333
7,TCGA-05-4395,male,76.0,IIIB,Indivumed,7.700000
8,TCGA-05-4396,male,76.0,IIIB,Indivumed,43.266667
9,TCGA-05-4397,male,65.0,IIB,Indivumed,24.866667


### Gene counts

The SQL keeps every patient and all five genes. `Silent` mutations are excluded. A count of `0` means no qualifying mutation was found. It does not mean the value is missing.

In [4]:
gene_counts_long = q("""
WITH target_genes(gene_symbol) AS (
    VALUES ('TP53'),
           ('KRAS'),
           ('EGFR'),
           ('STK11'),
           ('KEAP1')
)
SELECT
    p.patient_id,
    tg.gene_symbol,
    COUNT(m.mutation_id)
        AS protein_altering_mutation_count
FROM patients AS p
CROSS JOIN target_genes AS tg
LEFT JOIN samples AS s
  ON s.patient_id = p.patient_id
LEFT JOIN mutations AS m
  ON m.sample_id = s.sample_id
 AND m.gene_symbol = tg.gene_symbol
 AND m.variant_classification <> 'Silent'
GROUP BY
    p.patient_id,
    tg.gene_symbol
ORDER BY
    p.patient_id,
    tg.gene_symbol
""")

print("Expected rows:", 566 * 5)
print("Actual rows:", len(gene_counts_long))
print(
    "Unique patients:",
    gene_counts_long["patient_id"].nunique()
)

gene_counts_long.head(10)

Expected rows: 2830
Actual rows: 2830
Unique patients: 566


,patient_id,gene_symbol,protein_altering_mutation_count
0,TCGA-05-4244,EGFR,0
1,TCGA-05-4244,KEAP1,0
2,TCGA-05-4244,KRAS,1
3,TCGA-05-4244,STK11,0
4,TCGA-05-4244,TP53,0
5,TCGA-05-4249,EGFR,0
6,TCGA-05-4249,KEAP1,0
7,TCGA-05-4249,KRAS,1
8,TCGA-05-4249,STK11,0
9,TCGA-05-4249,TP53,0


In [5]:
gene_counts_wide = gene_counts_long.pivot(
    index="patient_id",
    columns="gene_symbol",
    values="protein_altering_mutation_count"
).reset_index()

gene_counts_wide.columns.name = None

target_genes = ["TP53", "KRAS", "EGFR", "STK11", "KEAP1"]

rename_counts = {
    gene: f"{gene}_protein_altering_count"
    for gene in target_genes
}

gene_counts_wide = gene_counts_wide.rename(
    columns=rename_counts
)

gene_count_columns = [
    f"{gene}_protein_altering_count"
    for gene in target_genes
]

gene_counts_wide = gene_counts_wide[
    ["patient_id"] + gene_count_columns
]

print("Wide gene-count rows:", len(gene_counts_wide))
gene_counts_wide.head(10)

Wide gene-count rows: 566


,patient_id,TP53_protein_altering_count,KRAS_protein_altering_count,EGFR_protein_altering_count,STK11_protein_altering_count,KEAP1_protein_altering_count
0,TCGA-05-4244,0,1,0,0,0
1,TCGA-05-4249,0,1,0,0,0
2,TCGA-05-4250,0,1,0,0,0
3,TCGA-05-4382,1,0,2,0,0
4,TCGA-05-4384,1,0,0,0,1
5,TCGA-05-4389,0,0,0,0,1
6,TCGA-05-4390,0,1,0,0,0
7,TCGA-05-4395,1,1,0,1,1
8,TCGA-05-4396,1,0,0,0,0
9,TCGA-05-4397,1,0,0,0,0


### Wide table and merge

Python reshapes the long gene counts into five columns. The left merge starts from the patient table. This keeps patients even when a clinical value is missing.

In [6]:
feature_table = patient_features.merge(
    gene_counts_wide,
    on="patient_id",
    how="left",
    validate="one_to_one"
)

print("Patients before merge:", len(patient_features))
print("Patients after merge:", len(feature_table))
print(
    "Unique patients after merge:",
    feature_table["patient_id"].nunique()
)

assert len(feature_table) == 566
assert feature_table["patient_id"].is_unique

Patients before merge: 566
Patients after merge: 566
Unique patients after merge: 566


### Missing values

The table has 566 patients before and after the merge. No patient was lost. Only gene-count columns are filled with `0`. Clinical gaps remain `NA` because those values were not recorded.

In [7]:
feature_table[gene_count_columns] = (
    feature_table[gene_count_columns]
    .fillna(0)
    .astype(int)
)

In [8]:
feature_table[
    [
        "sex",
        "age_at_diagnosis",
        "stage",
        "tissue_source_site",
        "tmb_nonsynonymous",
    ]
].isna().sum()

sex                   52
age_at_diagnosis      71
stage                 54
tissue_source_site     0
tmb_nonsynonymous      0
dtype: int64

### Missing-data check

The final table keeps 52 missing sex values, 71 missing age values, and 54 missing stage values. Tissue source site and tumor mutation burden have no missing values.

In [9]:
feature_table.head(10)

,patient_id,sex,age_at_diagnosis,stage,tissue_source_site,tmb_nonsynonymous,TP53_protein_altering_count,KRAS_protein_altering_count,EGFR_protein_altering_count,STK11_protein_altering_count,KEAP1_protein_altering_count
0,TCGA-05-4244,male,70.0,IV,Indivumed,6.400000,0,1,0,0,0
1,TCGA-05-4249,male,67.0,IB,Indivumed,10.000000,0,1,0,0,0
2,TCGA-05-4250,female,79.0,IIIA,Indivumed,10.500000,0,1,0,0,0
3,TCGA-05-4382,male,68.0,IB,Indivumed,51.733333,1,0,2,0,0
4,TCGA-05-4384,male,66.0,IIIA,Indivumed,3.966667,1,0,0,0,1
5,TCGA-05-4389,male,70.0,IA,Indivumed,7.033333,0,0,0,0,1
6,TCGA-05-4390,female,58.0,IB,Indivumed,17.333333,0,1,0,0,0
7,TCGA-05-4395,male,76.0,IIIB,Indivumed,7.700000,1,1,0,1,1
8,TCGA-05-4396,male,76.0,IIIB,Indivumed,43.266667,1,0,0,0,0
9,TCGA-05-4397,male,65.0,IIB,Indivumed,24.866667,1,0,0,0,0


### Output

The TSV writes unrecorded clinical values as `NA`. Gene counts remain numeric. A gene count of `0` means no qualifying mutation was found.

In [10]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / "feature_table.tsv"

feature_table.to_csv(
    output_path,
    sep="\t",
    index=False,
    na_rep="NA"
)

print("Saved:", output_path.resolve())

Saved: /Users/sofia/Desktop/hds-practical/Lab4/outputs/feature_table.tsv


### Task 5: Data dictionary

This metadata step is additional Python/pandas code. It is not taken directly from Week 8.

The dictionary has one row for each final feature-table column. It records the meaning, units, source table, and missing-value rule.

Study context from cBioPortal: this is the Lung Adenocarcinoma TCGA PanCancer Atlas study. The page reports 566 patients and 566 samples. Mutation data are available for all 566 samples (100%).

Source: [cBioPortal study page](https://www.cbioportal.org/study/summary?id=luad_tcga_pan_can_atlas_2018)

In [11]:
data_dictionary = pd.DataFrame([
    {
        "column_name": "patient_id",
        "meaning": "Unique patient identifier in the TCGA Lung Adenocarcinoma PanCancer Atlas cohort",
        "units": "identifier",
        "source_table": "patients",
        "missing_value_handling": "Not expected; this is the patient key.",
    },
    {
        "column_name": "sex",
        "meaning": "Patient sex from the study's clinical data",
        "units": "category",
        "source_table": "patients",
        "missing_value_handling": "NA means not recorded.",
    },
    {
        "column_name": "age_at_diagnosis",
        "meaning": "Age at diagnosis from the study's clinical data",
        "units": "years",
        "source_table": "patients",
        "missing_value_handling": "NA means not recorded.",
    },
    {
        "column_name": "stage",
        "meaning": "American Joint Committee on Cancer (AJCC) tumor stage code",
        "units": "category",
        "source_table": "patients",
        "missing_value_handling": "NA means not recorded.",
    },
    {
        "column_name": "tissue_source_site",
        "meaning": "TCGA code for the site that supplied the tumor sample",
        "units": "category",
        "source_table": "samples",
        "missing_value_handling": "NA means not recorded.",
    },
    {
        "column_name": "tmb_nonsynonymous",
        "meaning": "Nonsynonymous tumor mutation burden for the tumor sample",
        "units": "mutations per megabase",
        "source_table": "samples",
        "missing_value_handling": "NA means not recorded.",
    },
    {
        "column_name": "TP53_protein_altering_count",
        "meaning": "Number of non-Silent TP53 mutations in the patient's tumor sample",
        "units": "count",
        "source_table": "mutations (joined through samples)",
        "missing_value_handling": "0 means no qualifying mutation; Silent mutations are excluded.",
    },
    {
        "column_name": "KRAS_protein_altering_count",
        "meaning": "Number of non-Silent KRAS mutations in the patient's tumor sample",
        "units": "count",
        "source_table": "mutations (joined through samples)",
        "missing_value_handling": "0 means no qualifying mutation; Silent mutations are excluded.",
    },
    {
        "column_name": "EGFR_protein_altering_count",
        "meaning": "Number of non-Silent EGFR mutations in the patient's tumor sample",
        "units": "count",
        "source_table": "mutations (joined through samples)",
        "missing_value_handling": "0 means no qualifying mutation; Silent mutations are excluded.",
    },
    {
        "column_name": "STK11_protein_altering_count",
        "meaning": "Number of non-Silent STK11 mutations in the patient's tumor sample",
        "units": "count",
        "source_table": "mutations (joined through samples)",
        "missing_value_handling": "0 means no qualifying mutation; Silent mutations are excluded.",
    },
    {
        "column_name": "KEAP1_protein_altering_count",
        "meaning": "Number of non-Silent KEAP1 mutations in the patient's tumor sample",
        "units": "count",
        "source_table": "mutations (joined through samples)",
        "missing_value_handling": "0 means no qualifying mutation; Silent mutations are excluded.",
    },
])

expected_columns = list(feature_table.columns)
assert len(data_dictionary) == len(expected_columns)
assert data_dictionary["column_name"].tolist() == expected_columns

print("Feature-table columns:", len(expected_columns))
print("Data-dictionary rows:", len(data_dictionary))
display(data_dictionary)

data_dictionary_path = output_dir / "data_dictionary.tsv"
data_dictionary.to_csv(data_dictionary_path, sep="\t", index=False)
print("Saved:", data_dictionary_path.resolve())

Feature-table columns: 11
Data-dictionary rows: 11


,column_name,meaning,units,source_table,missing_value_handling
0,patient_id,Unique patient identifier in the TCGA Lung Ade...,identifier,patients,Not expected; this is the patient key.
1,sex,Patient sex from the study's clinical data,category,patients,NA means not recorded.
2,age_at_diagnosis,Age at diagnosis from the study's clinical data,years,patients,NA means not recorded.
3,stage,American Joint Committee on Cancer (AJCC) tumo...,category,patients,NA means not recorded.
4,tissue_source_site,TCGA code for the site that supplied the tumor...,category,samples,NA means not recorded.
5,tmb_nonsynonymous,Nonsynonymous tumor mutation burden for the tu...,mutations per megabase,samples,NA means not recorded.
6,TP53_protein_altering_count,Number of non-Silent TP53 mutations in the pat...,count,mutations (joined through samples),0 means no qualifying mutation; Silent mutatio...
7,KRAS_protein_altering_count,Number of non-Silent KRAS mutations in the pat...,count,mutations (joined through samples),0 means no qualifying mutation; Silent mutatio...
8,EGFR_protein_altering_count,Number of non-Silent EGFR mutations in the pat...,count,mutations (joined through samples),0 means no qualifying mutation; Silent mutatio...
9,STK11_protein_altering_count,Number of non-Silent STK11 mutations in the pa...,count,mutations (joined through samples),0 means no qualifying mutation; Silent mutatio...


Saved: /Users/sofia/Desktop/hds-practical/Lab4/outputs/data_dictionary.tsv


All 11 feature-table columns appear once. The descriptions use the cBioPortal study context. Clinical NA values mean not recorded. Gene-count zero means no qualifying non-Silent mutation was found; it does not mean missing.